In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error = pd.read_csv('error_logs.csv')

# Parse timestamps as UTC datetimes
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)
df_error['timestamp'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)

# Aggregate helper: compute count, min/max timestamp, mean, max, P90, P95
metric_agg = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .agg(
        count=('value', 'count'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
        mean_value=('value', 'mean'),
        max_value=('value', 'max'),
        p90_value=('value', lambda x: x.quantile(0.9)),
        p95_value=('value', lambda x: x.quantile(0.95)),
    )
    .reset_index()
    .sort_values('count', ascending=False)
)
metric_agg_top20 = metric_agg.head(20)

trace_agg = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], dropna=False)
    .agg(
        count=('value', 'count'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
        mean_value=('value', 'mean'),
        max_value=('value', 'max'),
        p90_value=('value', lambda x: x.quantile(0.9)),
        p95_value=('value', lambda x: x.quantile(0.95)),
    )
    .reset_index()
    .sort_values('count', ascending=False)
)
trace_agg_top20 = trace_agg.head(20)

log_agg = (
    df_log
    .groupby(['cmdb_id', 'log_name'], dropna=False)
    .agg(
        count=('value', 'count'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
        mean_value=('value', 'mean'),
        max_value=('value', 'max'),
        p90_value=('value', lambda x: x.quantile(0.9)),
        p95_value=('value', lambda x: x.quantile(0.95)),
    )
    .reset_index()
    .sort_values('count', ascending=False)
)
log_agg_top20 = log_agg.head(20)

# Unique cmdb_id lists per file
unique_metric_cmdb = pd.Series(sorted(df_metric['cmdb_id'].dropna().unique()), name='metric_cmdb_ids')
unique_trace_cmdb = pd.Series(sorted(df_trace['cmdb_id'].dropna().unique()), name='trace_cmdb_ids')
unique_log_cmdb = pd.Series(sorted(df_log['cmdb_id'].dropna().unique()), name='log_cmdb_ids')

# Compose a compact DataFrame summarizing unique cmdb_ids per file
unique_cmdb_df = pd.DataFrame({
    'file': ['metric.csv', 'trace.csv', 'log.csv'],
    'unique_cmdb_ids': [
        list(unique_metric_cmdb.values),
        list(unique_trace_cmdb.values),
        list(unique_log_cmdb.values),
    ]
})

# Sample rows (up to first 10) for schema verification
metric_sample = df_metric.head(10)
trace_sample = df_trace.head(10)
log_sample = df_log.head(10)
error_sample = df_error.head(10)

# Display the requested compact outputs (variables separated by commas)
metric_agg_top20, trace_agg_top20, log_agg_top20, unique_cmdb_df, metric_sample, trace_sample, log_sample, error_sample

```
Out[1]:
```
summary = (
    "Summary of telemetry aggregates (plain English):\n"
    "- All three telemetry files (metric, trace, log) contain 25 samples per series covering 2024-01-24 22:47:00 to 2024-01-24 23:11:00 UTC (consistent sampling window).\n"
    "- Metrics: common KPIs include cpu, mem, diskio, socket, latency-50/90, workload and error. Notable items: ts-travel2-service shows relatively high CPU (mean ~4.6, max ~5.2), high socket counts (mean ~17.1, max ~19.3) and high memory (~2.74e8). ts-admin-basic-info-service and others have expected lower CPU and moderate disk I/O; ts-voucher-service socket is constant at 4.\n"
    "- Traces: many \"trace.to_...\" and \"trace.from_...\" series are present (including a top-level 'root' cmdb_id). Several high row_count series exist (for example trace.from_ts-travel2-service row_count mean ~1823, max ~2681; trace.to_ts-route-service row_count mean ~418, max ~599). Most trace error_rate series are 0.\n"
    "- Logs: log.row_count is large for some services (e.g., ts-basic-service mean ~2395, ts-order-related services hundreds of rows per minute). log.error_count is 0 for the top aggregated services shown.\n"
    "- Unique cmdb_id lists were returned for metric/trace/log and sample rows (up to 10) show parsed UTC timestamps and expected columns: metric(timestamp, cmdb_id, kpi_name, value), trace(timestamp, cmdb_id, trace_name, value), log(timestamp, cmdb_id, log_name, value), error_logs(timestamp, cmdb_id, message).\n"
    "- Overall interpretation: telemetry is well-formed and shows no widespread log or trace error counts in the sampled window. Resource/telemetry hotspots to investigate further (if needed) are ts-travel2-service (CPU, socket, mem, disk I/O) and services with very high log.row_count (e.g., ts-basic-service, ts-order-service)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                         cmdb_id    kpi_name  count             min_timestamp             max_timestamp    mean_value     max_value     p90_value     p95_value
362           ts-voucher-service      socket     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00  4.000000e+00  4.000000e+00  4.000000e+00  4.000000e+00
0    ts-admin-basic-info-service         cpu     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00  5.839211e-01  3.021012e+00  7.681216e-01  1.302989e+00
1    ts-admin-basic-info-service      diskio     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00  5.282981e+04  6.688626e+04  5.800166e+04  5.846331e+04
2    ts-admin-basic-info-service       error     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
3    ts-admin-basic-info-service  latency-50     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00  1.642993e-02  1.792070e-02  1.699548e-02  1.721221e-02
4    ts-admin-basic-info-service  latency-90     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00  2.439420e-02  5.116226e-02  2.342687e-02  2.351649e-02
5    ts-admin-basic-info-service         mem     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00  2.346537e+08  2.356707e+08  2.354136e+08  2.356199e+08
6    ts-admin-basic-info-service      socket     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00  6.368429e+00  7.150000e+00  6.813333e+00  6.886667e+00
7    ts-admin-basic-info-service    workload     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00  6.225148e-01  7.121667e-01  6.857967e-01  7.025967e-01
8         ts-admin-order-service         cpu     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00  9.722943e-02  1.101294e-01  9.987370e-02  1.054757e-01
9         ts-admin-order-service      diskio     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00  4.027173e+04  4.909870e+04  4.605051e+04  4.773335e+04
323             ts-travel2-mongo         mem     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00  6.726826e+07  6.737101e+07  6.736055e+07  6.736940e+07
324             ts-travel2-mongo      socket     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00  6.000000e+00  6.000000e+00  6.000000e+00  6.000000e+00
325           ts-travel2-service         cpu     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00  4.620519e+00  5.211556e+00  5.080768e+00  5.106510e+00
326           ts-travel2-service      diskio     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00  4.938706e+04  5.729876e+04  5.621084e+04  5.647427e+04
327           ts-travel2-service  latency-50     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00  1.759431e-02  1.816356e-02  1.802680e-02  1.804184e-02
328           ts-travel2-service  latency-90     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00  3.697062e-01  3.739973e-01  3.716945e-01  3.717993e-01
329           ts-travel2-service         mem     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00  2.735951e+08  2.737874e+08  2.737544e+08  2.737593e+08
330           ts-travel2-service      socket     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00  1.715433e+01  1.931667e+01  1.843333e+01  1.866667e+01
315            ts-travel-service       error     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00,                 cmdb_id                                         trace_name  count             min_timestamp             max_timestamp   mean_value    max_value    p90_value    p95_value
871     ts-user-service                 trace.to_ts-user-service.row_count     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00     9.920000    20.000000    15.600000    19.200000
0                  root  trace.to_ts-admin-basic-info-service.duration_...     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00     0.010118     0.013452     0.010941     0.012569
1                  root  trace.to_ts-admin-basic-info-service.duration_p95     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00     0.013157     0.025390     0.016270     0.017323
2                  root    trace.to_ts-admin-basic-info-service.error_rate     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00     0.000000     0.000000     0.000000     0.000000
3                  root     trace.to_ts-admin-basic-info-service.row_count     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00    17.560000    21.000000    20.000000    20.800000
4                  root     trace.to_ts-admin-travel-service.duration_mean     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00     0.173762     0.268924     0.242451     0.246060
5                  root      trace.to_ts-admin-travel-service.duration_p95     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00     0.217735     0.319981     0.313433     0.317493
6                  root        trace.to_ts-admin-travel-service.error_rate     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00     0.000000     0.000000     0.000000     0.000000
832  ts-travel2-service            trace.to_ts-route-service.duration_mean     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00     0.003030     0.003219     0.003109     0.003191
833  ts-travel2-service             trace.to_ts-route-service.duration_p95     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00     0.003821     0.004519     0.004035     0.004171
834  ts-travel2-service               trace.to_ts-route-service.error_rate     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00     0.000000     0.000000     0.000000     0.000000
835  ts-travel2-service                trace.to_ts-route-service.row_count     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00   418.000000   599.000000   562.400000   564.000000
836  ts-travel2-service             trace.to_ts-seat-service.duration_mean     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00     0.044324     0.049010     0.045245     0.046439
837  ts-travel2-service              trace.to_ts-seat-service.duration_p95     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00     0.054430     0.057820     0.057150     0.057559
838  ts-travel2-service                trace.to_ts-seat-service.error_rate     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00     0.000000     0.000000     0.000000     0.000000
839  ts-travel2-service                 trace.to_ts-seat-service.row_count     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00   111.440000   161.000000   149.600000   153.200000
824  ts-travel2-service        trace.from_ts-travel2-service.duration_mean     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00     0.009722     0.010765     0.009926     0.009990
825  ts-travel2-service         trace.from_ts-travel2-service.duration_p95     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00     0.046520     0.048993     0.047439     0.047711
826  ts-travel2-service           trace.from_ts-travel2-service.error_rate     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00     0.000000     0.000000     0.000000     0.000000
827  ts-travel2-service            trace.from_ts-travel2-service.row_count     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00  1823.440000  2681.000000  2479.200000  2515.200000,                         cmdb_id         log_name  count             min_timestamp             max_timestamp  mean_value  max_value  p90_value  p95_value
0   ts-admin-basic-info-service  log.error_count     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00        0.00          0        0.0        0.0
1   ts-admin-basic-info-service    log.row_count     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00       26.84         33       31.2       32.0
2       ts-admin-travel-service  log.error_count     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00        0.00          0        0.0        0.0
3       ts-admin-travel-service    log.row_count     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00       26.76         36       30.0       31.6
7          ts-assurance-service    log.row_count     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00       10.64         26       14.2       18.2
6          ts-assurance-service  log.error_count     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00        0.00          0        0.0        0.0
13             ts-basic-service    log.row_count     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00     2395.16       2860     2667.8     2747.0
12             ts-basic-service  log.error_count     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00        0.00          0        0.0        0.0
11              ts-auth-service    log.row_count     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00      312.08        364      338.4      344.8
10              ts-auth-service  log.error_count     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00        0.00          0        0.0        0.0
25          ts-contacts-service    log.row_count     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00       39.84         90       55.2       60.0
24          ts-contacts-service  log.error_count     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00        0.00          0        0.0        0.0
20           ts-consign-service  log.error_count     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00        0.00          0        0.0        0.0
21           ts-consign-service    log.row_count     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00       45.64         77       70.0       70.0
70          ts-security-service  log.error_count     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00        0.00          0        0.0        0.0
71          ts-security-service    log.row_count     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00       53.12        120       73.6       80.0
47             ts-order-service    log.row_count     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00      375.76        509      461.8      469.4
46             ts-order-service  log.error_count     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00        0.00          0        0.0        0.0
45       ts-order-other-service    log.row_count     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00      426.60        562      493.8      513.0
44       ts-order-other-service  log.error_count     25 2024-01-24 22:47:00+00:00 2024-01-24 23:11:00+00:00        0.00          0        0.0        0.0,          file                                    unique_cmdb_ids
0  metric.csv  [ts-admin-basic-info-service, ts-admin-order-s...
1   trace.csv  [root, ts-admin-basic-info-service, ts-admin-t...
2     log.csv  [ts-admin-basic-info-service, ts-admin-travel-...,                   timestamp                      cmdb_id    kpi_name         value
0 2024-01-24 22:47:00+00:00  ts-admin-basic-info-service         cpu  3.466676e-01
1 2024-01-24 22:47:00+00:00  ts-admin-basic-info-service      diskio  5.714337e+04
2 2024-01-24 22:47:00+00:00  ts-admin-basic-info-service       error  0.000000e+00
3 2024-01-24 22:47:00+00:00  ts-admin-basic-info-service  latency-50  1.682254e-02
4 2024-01-24 22:47:00+00:00  ts-admin-basic-info-service  latency-90  2.336451e-02
5 2024-01-24 22:47:00+00:00  ts-admin-basic-info-service         mem  2.334833e+08
6 2024-01-24 22:47:00+00:00  ts-admin-basic-info-service      socket  6.160714e+00
7 2024-01-24 22:47:00+00:00  ts-admin-basic-info-service    workload  6.072857e-01
8 2024-01-24 22:47:00+00:00       ts-admin-order-service         cpu  9.522613e-02
9 2024-01-24 22:47:00+00:00       ts-admin-order-service      diskio  4.155942e+04,                   timestamp cmdb_id                                         trace_name      value
0 2024-01-24 22:47:00+00:00    root  trace.to_ts-admin-basic-info-service.duration_...   0.010188
1 2024-01-24 22:47:00+00:00    root  trace.to_ts-admin-basic-info-service.duration_p95   0.011906
2 2024-01-24 22:47:00+00:00    root    trace.to_ts-admin-basic-info-service.error_rate   0.000000
3 2024-01-24 22:47:00+00:00    root     trace.to_ts-admin-basic-info-service.row_count  18.000000
4 2024-01-24 22:47:00+00:00    root     trace.to_ts-admin-travel-service.duration_mean   0.125654
5 2024-01-24 22:47:00+00:00    root      trace.to_ts-admin-travel-service.duration_p95   0.136410
6 2024-01-24 22:47:00+00:00    root        trace.to_ts-admin-travel-service.error_rate   0.000000
7 2024-01-24 22:47:00+00:00    root         trace.to_ts-admin-travel-service.row_count   8.000000
8 2024-01-24 22:47:00+00:00    root        trace.to_ts-assurance-service.duration_mean   0.003723
9 2024-01-24 22:47:00+00:00    root         trace.to_ts-assurance-service.duration_p95   0.004318,                   timestamp                      cmdb_id         log_name  value
0 2024-01-24 22:47:00+00:00  ts-admin-basic-info-service  log.error_count      0
1 2024-01-24 22:47:00+00:00  ts-admin-basic-info-service    log.row_count     27
2 2024-01-24 22:47:00+00:00      ts-admin-travel-service  log.error_count      0
3 2024-01-24 22:47:00+00:00      ts-admin-travel-service    log.row_count     24
4 2024-01-24 22:47:00+00:00           ts-assurance-mongo  log.error_count      0
5 2024-01-24 22:47:00+00:00           ts-assurance-mongo    log.row_count      1
6 2024-01-24 22:47:00+00:00         ts-assurance-service  log.error_count      0
7 2024-01-24 22:47:00+00:00         ts-assurance-service    log.row_count     11
8 2024-01-24 22:47:00+00:00                ts-auth-mongo  log.error_count      0
9 2024-01-24 22:47:00+00:00                ts-auth-mongo    log.row_count      1, Empty DataFrame
Columns: [timestamp, cmdb_id, message, error]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse existing df_metric (assumed loaded and timestamp-parsed in UTC)
# If not present, load and parse
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Target components
targets = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# Restrict to target cmdb_ids
df_m_targets = df_metric[df_metric['cmdb_id'].isin(targets)].copy()

# Ensure value is numeric
df_m_targets['value'] = pd.to_numeric(df_m_targets['value'], errors='coerce')

# 1) Global aggregates (computed on full available series per (cmdb_id,kpi_name))
agg_full = (
    df_m_targets
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(
        count_full='count',
        mean_full='mean',
        max_full='max',
        p90=lambda s: s.quantile(0.90),
        p95=lambda s: s.quantile(0.95),
    )
    .reset_index()
)

# 2) Filter original metric rows to incident window and compute breaches against global p95
window_start = pd.to_datetime("2024-01-24 22:44:04", utc=True)
window_end   = pd.to_datetime("2024-01-24 23:14:04", utc=True)

df_window = df_m_targets[(df_m_targets['timestamp'] >= window_start) & (df_m_targets['timestamp'] <= window_end)].copy()

# Merge global p95 into window rows so decision uses global threshold
df_window = df_window.merge(agg_full[['cmdb_id','kpi_name','p95']], on=['cmdb_id','kpi_name'], how='left')

# Mark breaches where value >= global p95 (note: p95 may be NaN if group had no data)
df_window['is_breach'] = (df_window['value'] >= df_window['p95'])

# Aggregate breach info per group
breach_agg = (
    df_window[df_window['is_breach']]
    .groupby(['cmdb_id','kpi_name'], dropna=False)
    .agg(
        count_breach_in_window=('is_breach','sum'),
        earliest_breach_timestamp_in_window=('timestamp','min'),
        max_breach_value_in_window=('value','max'),
    )
    .reset_index()
)

# Ensure groups with zero breaches are included: left-join agg_full with breach_agg
result = agg_full.merge(breach_agg, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for non-breaching groups
result['count_breach_in_window'] = result['count_breach_in_window'].fillna(0).astype(int)
result['earliest_breach_timestamp_in_window'] = result['earliest_breach_timestamp_in_window'].where(result['count_breach_in_window']>0, pd.NaT)
result['max_breach_value_in_window'] = result['max_breach_value_in_window'].where(result['count_breach_in_window']>0, np.nan)

# Order columns as requested and sort by count_breach_in_window desc then count_full desc
result = result[
    ['cmdb_id','kpi_name','count_full','mean_full','max_full','p90','p95',
     'count_breach_in_window','earliest_breach_timestamp_in_window','max_breach_value_in_window']
]

result_sorted = result.sort_values(['count_breach_in_window','count_full'], ascending=[False, False]).reset_index(drop=True)

# Limit to top 50 rows
top50 = result_sorted.head(50)

# Display the compact result
top50

```
Out[2]:
```
summary = (
    "Summary of metric breach analysis for the five target services (UTC):\n"
    "- Total groups analyzed: 37 (each is a (cmdb_id, kpi_name) pair).\n"
    "- Strongest/biggest breach counts:\n"
    "  * ts-order-service, kpi 'error': count_full=25 and count_breach_in_window=25 (p95=0.0). All sampled points equal the global P95 (0), meaning no non‑zero errors but values meet the threshold of 0.\n"
    "  * ts-travel-service, kpi 'error': same situation (count_breach_in_window=25, p95=0.0).\n"
    "- Notable resource/telemetry breaches (non-zero metrics):\n"
    "  * ts-route-service, 'socket': 3 points in the incident window >= global P95; earliest breach 2024-01-24T22:56:00Z, max breach value 12.7.\n"
    "  * ts-auth-service, 'cpu': 2 breaches; earliest 2024-01-24T23:01:00Z, max breach value ~15.715.\n"
    "  * ts-auth-service, 'diskio': 2 breaches; earliest 2024-01-24T22:51:00Z, max breach value ~64792.8.\n"
    "  * ts-travel-service shows a few resource KPIs (mem, socket, workload, latency-50/90) with 2 breaches each; earliest breach timestamps are around 2024-01-24T23:00–23:08:00Z.\n"
    "- Interpretation and recommendation:\n"
    "  * The 'error' series for ts-order-service and ts-travel-service have p95=0, so every sample meets >= p95 but this does not indicate active errors >0; it simply reflects the threshold calculation. Treat those as non-actionable unless error count >0 appears.\n"
    "  * Focus investigation on the non-zero metric breaches: ts-route-service socket spikes and ts-auth-service CPU/diskio spikes are the clearest outliers during the incident window. ts-travel-service resource metrics also warrant review.\n"
    "  * Next steps: inspect logs/traces around the listed breach timestamps for those services, and monitor subsequent minutes to confirm persistence or recurrence."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  count_full     mean_full      max_full           p90           p95  count_breach_in_window earliest_breach_timestamp_in_window  max_breach_value_in_window
0    ts-order-service       error          25  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00                      25           2024-01-24 22:47:00+00:00                0.000000e+00
1   ts-travel-service       error          25  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00                      25           2024-01-24 22:47:00+00:00                0.000000e+00
2    ts-route-service      socket          25  1.216086e+01  1.270000e+01  1.259333e+01  1.260000e+01                       3           2024-01-24 22:56:00+00:00                1.270000e+01
3     ts-auth-service         cpu          25  1.394014e+01  1.571544e+01  1.473150e+01  1.514600e+01                       2           2024-01-24 23:01:00+00:00                1.571544e+01
4     ts-auth-service      diskio          25  5.150188e+04  6.479280e+04  6.065994e+04  6.147855e+04                       2           2024-01-24 22:51:00+00:00                6.479280e+04
..                ...         ...         ...           ...           ...           ...           ...                     ...                                 ...                         ...
32  ts-travel-service  latency-50          25  3.246904e-02  5.714195e-02  5.634166e-02  5.667795e-02                       2           2024-01-24 23:02:00+00:00                5.714195e-02
33  ts-travel-service  latency-90          25  2.517864e-01  5.466716e-01  4.086007e-01  4.816498e-01                       2           2024-01-24 23:02:00+00:00                5.466716e-01
34  ts-travel-service         mem          25  2.903396e+08  2.927605e+08  2.927229e+08  2.927369e+08                       2           2024-01-24 23:08:00+00:00                2.927605e+08
35  ts-travel-service      socket          25  1.563800e+01  2.173333e+01  1.819667e+01  1.997000e+01                       2           2024-01-24 23:00:00+00:00                2.173333e+01
36  ts-travel-service    workload          25  9.320578e+00  1.201665e+01  1.119450e+01  1.167936e+01                       2           2024-01-24 23:00:00+00:00                1.201665e+01

[37 rows x 10 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse df_trace if available; otherwise load and parse timestamps as UTC
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')
    df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Ensure timestamp parsed and value numeric
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# Target cmdb_ids (including 'root')
targets = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service","root"]

df_tgt = df_trace[df_trace['cmdb_id'].isin(targets)].copy()

# 1) Compute full-series aggregates per (cmdb_id, trace_name)
agg_full_trace = (
    df_tgt
    .groupby(['cmdb_id','trace_name'], dropna=False)['value']
    .agg(
        count_full='count',
        mean_full='mean',
        max_full='max',
        p90=lambda s: s.quantile(0.90),
        p95=lambda s: s.quantile(0.95),
    )
    .reset_index()
)

# 2) Filter to incident window
window_start = pd.to_datetime("2024-01-24 22:44:04", utc=True)
window_end   = pd.to_datetime("2024-01-24 23:14:04", utc=True)

df_window_trace = df_tgt[(df_tgt['timestamp'] >= window_start) & (df_tgt['timestamp'] <= window_end)].copy()

# Merge global p95/p90 info into window rows
df_window_trace = df_window_trace.merge(agg_full_trace[['cmdb_id','trace_name','p95']], on=['cmdb_id','trace_name'], how='left')

# Identify error_rate traces
df_window_trace['is_error_rate'] = df_window_trace['trace_name'].str.contains('error_rate', case=False, na=False)

# Mark breaches: value >= p95 OR (error_rate and value > 0)
# Note: if p95 is NaN, the >= comparison yields False; error_rate check still applies
df_window_trace['is_breach'] = (df_window_trace['value'] >= df_window_trace['p95']) | (df_window_trace['is_error_rate'] & (df_window_trace['value'] > 0))

# Aggregate breach info per group
breach_agg_trace = (
    df_window_trace[df_window_trace['is_breach']]
    .groupby(['cmdb_id','trace_name'], dropna=False)
    .agg(
        count_breach_in_window=('is_breach','sum'),
        earliest_breach_timestamp_in_window=('timestamp','min'),
        max_breach_value_in_window=('value','max'),
    )
    .reset_index()
)

# Left-join full aggregates with breach aggregates
result_trace = agg_full_trace.merge(breach_agg_trace, on=['cmdb_id','trace_name'], how='left')

# Fill NaNs for non-breaching groups
result_trace['count_breach_in_window'] = result_trace['count_breach_in_window'].fillna(0).astype(int)
result_trace['earliest_breach_timestamp_in_window'] = result_trace['earliest_breach_timestamp_in_window'].where(result_trace['count_breach_in_window']>0, pd.NaT)
result_trace['max_breach_value_in_window'] = result_trace['max_breach_value_in_window'].where(result_trace['count_breach_in_window']>0, np.nan)

# Select and order columns as requested
result_trace = result_trace[
    ['cmdb_id','trace_name','count_full','mean_full','max_full','p90','p95',
     'count_breach_in_window','earliest_breach_timestamp_in_window','max_breach_value_in_window']
]

# Sort by breach count desc then count_full desc, limit to top 50
result_trace_sorted = result_trace.sort_values(['count_breach_in_window','count_full'], ascending=[False, False]).reset_index(drop=True)
top50_trace = result_trace_sorted.head(50)

# Display the compact result
top50_trace

```
Out[3]:
```
Summary of trace analysis (plain English):

- Scope: traces for cmdb_ids ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service","root"] over the entire series and specifically inside 2024-01-24 22:44:04 → 2024-01-24 23:14:04 UTC.

- What the table shows: many trace series of type "error_rate" were flagged as "breaches" in the incident window because the computed global P95 for those series is 0.0 and the trace values in the window are also 0.0, so value >= P95 evaluates true. As a result count_breach_in_window equals count_full for many error_rate rows, and earliest_breach_timestamp_in_window is typically 2024-01-24T22:47:00Z (or similar).

- Important interpretation: these flagged "breaches" are not actionable errors — the error_rate values are zero (max_breach_value_in_window = 0.0). The P95=0 artifact causes equality to count as a breach. In short: no non-zero trace error rates were observed in the window.

- Non-error traces: the top portion of the results is dominated by error_rate artifacts. There is no clear evidence from the displayed top-50 of widespread non-zero trace anomalies (e.g., sustained elevated durations or error values) for these services in the incident window. If you need to detect real anomalies, use strict > P95 for numeric KPIs (or treat P95=0 specially) so zero-valued series do not produce misleading breach counts.

- Recommendation: focus investigation on services/timestamps flagged by independent signals (metrics/logs). From the metric analysis you ran earlier, candidates to inspect via traces around their breach timestamps include ts-auth-service (CPU/diskio spikes), ts-route-service (socket spikes) and ts-travel-service (resource spikes). Also review trace duration and row_count series for those services around the listed timestamps to confirm whether traces show latency or throughput issues.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                       trace_name  count_full  mean_full  max_full  p90  p95  count_breach_in_window earliest_breach_timestamp_in_window  max_breach_value_in_window
0                root  trace.to_ts-admin-basic-info-service.error_rate          25        0.0       0.0  0.0  0.0                      25           2024-01-24 22:47:00+00:00                         0.0
1                root      trace.to_ts-admin-travel-service.error_rate          25        0.0       0.0  0.0  0.0                      25           2024-01-24 22:47:00+00:00                         0.0
2                root         trace.to_ts-assurance-service.error_rate          25        0.0       0.0  0.0  0.0                      25           2024-01-24 22:47:00+00:00                         0.0
3                root              trace.to_ts-auth-service.error_rate          25        0.0       0.0  0.0  0.0                      25           2024-01-24 22:47:00+00:00                         0.0
4                root           trace.to_ts-consign-service.error_rate          25        0.0       0.0  0.0  0.0                      25           2024-01-24 22:47:00+00:00                         0.0
..                ...                                              ...         ...        ...       ...  ...  ...                     ...                                 ...                         ...
45   ts-order-service           trace.to_ts-station-service.error_rate          24        0.0       0.0  0.0  0.0                      24           2024-01-24 22:47:00+00:00                         0.0
46               root    trace.to_ts-preserve-other-service.error_rate          22        0.0       0.0  0.0  0.0                      22           2024-01-24 22:47:00+00:00                         0.0
47               root              trace.to_ts-seat-service.error_rate          22        0.0       0.0  0.0  0.0                      22           2024-01-24 22:47:00+00:00                         0.0
48  ts-travel-service        trace.from_ts-preserve-service.error_rate          22        0.0       0.0  0.0  0.0                      22           2024-01-24 22:47:00+00:00                         0.0
49               root             trace.to_ts-basic-service.error_rate          21        0.0       0.0  0.0  0.0                      21           2024-01-24 22:51:00+00:00                         0.0

[50 rows x 10 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse existing dataframes if available, otherwise load and parse timestamps as UTC
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')
    df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

try:
    df_error
except NameError:
    df_error = pd.read_csv('error_logs.csv')
    df_error['timestamp'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)

# Ensure timestamp columns are parsed and value is numeric for log
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# Target components and incident window (UTC)
targets = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]
window_start = pd.to_datetime("2024-01-24 22:44:04", utc=True)
window_end   = pd.to_datetime("2024-01-24 23:14:04", utc=True)

# Restrict logs to targets
df_log_tgt = df_log[df_log['cmdb_id'].isin(targets)].copy()

# 1) Full-series aggregates per (cmdb_id, log_name)
log_agg_full = (
    df_log_tgt
    .groupby(['cmdb_id','log_name'], dropna=False)['value']
    .agg(
        count_full='count',
        mean_full='mean',
        max_full='max',
        p90=lambda s: s.quantile(0.90),
        p95=lambda s: s.quantile(0.95),
    )
    .reset_index()
)

# 2) Filter to incident window and compute breaches against full-series p95
df_log_window = df_log_tgt[(df_log_tgt['timestamp'] >= window_start) & (df_log_tgt['timestamp'] <= window_end)].copy()

# Merge p95 into window rows (thresholds computed on full series)
df_log_window = df_log_window.merge(log_agg_full[['cmdb_id','log_name','p95']], on=['cmdb_id','log_name'], how='left')

# Mark breaches: value >= p95 (handles NaN p95 -> False)
df_log_window['is_breach'] = df_log_window['value'] >= df_log_window['p95']

# For log.error_count, also compute points with value > 0
df_log_window['is_error_gt0'] = (df_log_window['log_name']=='log.error_count') & (df_log_window['value'] > 0)

# Aggregate breach info per group from breached rows
breach_rows = df_log_window[df_log_window['is_breach']].copy()
breach_count = breach_rows.groupby(['cmdb_id','log_name']).size().reset_index(name='count_breach_in_window')
breach_earliest = breach_rows.groupby(['cmdb_id','log_name'])['timestamp'].min().reset_index(name='earliest_breach_timestamp_in_window')
breach_maxval = breach_rows.groupby(['cmdb_id','log_name'])['value'].max().reset_index(name='max_breach_value_in_window')

# Aggregate error_gt0 info for log.error_count
error_gt0_rows = df_log_window[df_log_window['is_error_gt0']].copy()
error_gt0_agg = error_gt0_rows.groupby(['cmdb_id','log_name']).agg(
    error_gt0_count=('is_error_gt0','sum'),
    earliest_error_gt0_timestamp_in_window=('timestamp','min')
).reset_index()

# Merge aggregated breach info into full agg
result_log = log_agg_full.merge(breach_count, on=['cmdb_id','log_name'], how='left')
result_log = result_log.merge(breach_earliest, on=['cmdb_id','log_name'], how='left')
result_log = result_log.merge(breach_maxval, on=['cmdb_id','log_name'], how='left')

# Fill NaNs for groups without breaches
result_log['count_breach_in_window'] = result_log['count_breach_in_window'].fillna(0).astype(int)
result_log['earliest_breach_timestamp_in_window'] = result_log['earliest_breach_timestamp_in_window'].where(result_log['count_breach_in_window']>0, pd.NaT)
result_log['max_breach_value_in_window'] = result_log['max_breach_value_in_window'].where(result_log['count_breach_in_window']>0, np.nan)

# Merge error_gt0 info (only relevant for log.error_count)
result_log = result_log.merge(error_gt0_agg, on=['cmdb_id','log_name'], how='left')
result_log['error_gt0_count'] = result_log['error_gt0_count'].fillna(0).astype(int)
result_log['earliest_error_gt0_timestamp_in_window'] = result_log['earliest_error_gt0_timestamp_in_window'].where(result_log['error_gt0_count']>0, pd.NaT)

# Select and order columns as requested
result_log = result_log[
    ['cmdb_id','log_name','count_full','mean_full','max_full','p90','p95',
     'count_breach_in_window','earliest_breach_timestamp_in_window','max_breach_value_in_window',
     'error_gt0_count','earliest_error_gt0_timestamp_in_window']
]

# Sort and limit to top 50
log_top50 = result_log.sort_values(['count_breach_in_window','count_full'], ascending=[False, False]).reset_index(drop=True).head(50)

# 2) error_logs: filter to targets and incident window, sample up to 20 rows for inspection
df_error['timestamp'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)
df_error_tgt_window = df_error[
    (df_error['cmdb_id'].isin(targets)) &
    (df_error['timestamp'] >= window_start) & (df_error['timestamp'] <= window_end)
].copy()

# Select required columns and limit to 20 rows
error_sample = df_error_tgt_window[['timestamp','cmdb_id','message']].head(20)

# Display compact outputs
log_top50, error_sample

```
Out[4]:
```
Summary of log analysis (UTC):

- What I computed: per (cmdb_id, log_name) full-series stats (count, mean, max, P90, P95) and how many points in the incident window 2024-01-24 22:44:04 → 2024-01-24 23:14:04 UTC meet value >= global P95. For log.error_count I also reported any points with value > 0 in the window. I also returned error_logs entries in the same window (up to 20 sample rows).

Key findings:

1) log.error_count
- For ts-auth-service, ts-order-service, ts-route-service, ts-travel-service:
  - count_full = 25, P95 = 0.0, count_breach_in_window = 25 (because value >= P95 with P95=0.0).
  - error_gt0_count = 0 and no earliest error>0 timestamps — there were no non‑zero error_count points in the window.
- Interpretation: the P95==0 artifact causes all zero-valued samples to appear as "breaches" by the >= rule. These are not actionable errors since error counts are zero.

2) log.row_count (notable spikes)
- ts-auth-service: count_full=25, mean~312.08, max=364; count_breach_in_window = 2, earliest breach 2024-01-24T22:56:00Z, max breach value in window = 364.
- ts-order-service: count_full=25, mean~375.76, max=509; count_breach_in_window = 2, earliest breach 2024-01-24T22:47:00Z, max breach value = 509.
- ts-route-service: count_full=25, mean~975.28, max=1130; count_breach_in_window = 2, earliest breach 2024-01-24T22:51:00Z, max breach value = 1130.
- ts-travel-service: count_full=25, mean~1685.00, max=2169; count_breach_in_window = 2, earliest breach 2024-01-24T22:51:00Z, max breach value = 2169.
- Interpretation: each of these services had a couple of minutes in the incident window where log.row_count reached or exceeded the global P95 — indicating short-lived increases in log volume/throughput.

3) error_logs.csv
- No matching error_log rows were found for the target services in the incident window (the sample returned is empty).

Recommendations:
- Do not treat the error_count P95==0 equality as an active error — there were no error_count > 0 events in the window.
- Investigate the services with log.row_count breaches (ts-auth-service, ts-order-service, ts-route-service, ts-travel-service) at the listed earliest timestamps (22:47, 22:51, 22:56 UTC) by:
  - Inspecting application logs around those timestamps for warnings/errors or sudden traffic spikes.
  - Correlating with metric/trace breaches you previously found (e.g., CPU/disk/socket spikes) to identify root cause.
- Since error_logs returned no textual error entries for these services in the window, focus first on throughput/latency/resource signals rather than explicit application exceptions.

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id         log_name  count_full  mean_full  max_full     p90     p95  count_breach_in_window earliest_breach_timestamp_in_window  max_breach_value_in_window  error_gt0_count earliest_error_gt0_timestamp_in_window
0    ts-auth-service  log.error_count          25       0.00         0     0.0     0.0                      25           2024-01-24 22:47:00+00:00                           0                0                                    NaT
1   ts-order-service  log.error_count          25       0.00         0     0.0     0.0                      25           2024-01-24 22:47:00+00:00                           0                0                                    NaT
2   ts-route-service  log.error_count          25       0.00         0     0.0     0.0                      25           2024-01-24 22:47:00+00:00                           0                0                                    NaT
3  ts-travel-service  log.error_count          25       0.00         0     0.0     0.0                      25           2024-01-24 22:47:00+00:00                           0                0                                    NaT
4    ts-auth-service    log.row_count          25     312.08       364   338.4   344.8                       2           2024-01-24 22:56:00+00:00                         364                0                                    NaT
5   ts-order-service    log.row_count          25     375.76       509   461.8   469.4                       2           2024-01-24 22:47:00+00:00                         509                0                                    NaT
6   ts-route-service    log.row_count          25     975.28      1130  1085.8  1101.4                       2           2024-01-24 22:51:00+00:00                        1130                0                                    NaT
7  ts-travel-service    log.row_count          25    1685.00      2169  2046.2  2088.2                       2           2024-01-24 22:51:00+00:00                        2169                0                                    NaT, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse df_metric if available; otherwise load and parse timestamps as UTC
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Ensure timestamp and value parsed correctly
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Target components
targets = ["ts-auth-service","ts-route-service","ts-travel-service","ts-order-service","ts-train-service"]

# 1) Compute global P95 per (cmdb_id, kpi_name) using full available series (only for target cmdb_ids)
agg_p95_full = (
    df_metric[df_metric['cmdb_id'].isin(targets)]
    .groupby(['cmdb_id','kpi_name'], dropna=False)['value']
    .agg(
        p95_full=lambda s: s.quantile(0.95)
    )
    .reset_index()
)

# 2) Filter metric rows to incident window
window_start = pd.to_datetime("2024-01-24 22:44:04", utc=True)
window_end   = pd.to_datetime("2024-01-24 23:14:04", utc=True)

df_window = df_metric[
    (df_metric['cmdb_id'].isin(targets)) &
    (df_metric['timestamp'] >= window_start) &
    (df_metric['timestamp'] <= window_end)
].copy()

# Merge p95 thresholds into window rows
df_window = df_window.merge(agg_p95_full, on=['cmdb_id','kpi_name'], how='left')

# Prepare container for per-group summaries
rows = []

# Process each (cmdb_id, kpi_name) present in the merged window (use groups from agg_p95_full to honor rule of threshold computed before filtering)
for _, grp in agg_p95_full.iterrows():
    cmdb = grp['cmdb_id']
    kpi = grp['kpi_name']
    p95_val = grp['p95_full']

    # Window rows for this group
    gdf = df_window[(df_window['cmdb_id']==cmdb) & (df_window['kpi_name']==kpi)].copy().sort_values('timestamp')
    count_points_in_window = len(gdf)
    if count_points_in_window == 0:
        # No points in window -> skip since we will only return groups with at least one point >= p95 in window
        continue

    # Compute boolean mask of points >= p95 (handle NaN p95 by treating as no-threshold -> no breaches)
    if pd.isna(p95_val):
        is_ge = np.zeros(count_points_in_window, dtype=bool)
    else:
        is_ge = (gdf['value'].to_numpy() >= p95_val)

    count_points_ge_p95_in_window = int(is_ge.sum())

    if count_points_ge_p95_in_window == 0:
        # skip groups without any breaches as requested
        continue

    # earliest timestamp where ge p95
    if count_points_ge_p95_in_window > 0:
        earliest_ts = gdf['timestamp'].iloc[np.where(is_ge)[0][0]]
    else:
        earliest_ts = pd.NaT

    # max value in window (overall, not just breaches)
    max_value_in_window = gdf['value'].max() if count_points_in_window>0 else np.nan

    # compute longest consecutive run of True in is_ge and its start timestamp
    arr = is_ge.astype(int)
    # pad with zeros to find edges
    padded = np.concatenate(([0], arr, [0]))
    diff = np.diff(padded)
    starts = np.where(diff==1)[0]  # indices in padded (which align to original indices)
    ends = np.where(diff==-1)[0]
    if len(starts)==0:
        longest_run = 0
        start_of_longest = pd.NaT
    else:
        lengths = ends - starts
        idx_max = int(np.argmax(lengths))
        longest_run = int(lengths[idx_max])
        start_idx = starts[idx_max]  # this is already the index into original array
        # map start_idx to timestamp in gdf
        try:
            start_of_longest = gdf['timestamp'].iloc[start_idx]
        except Exception:
            start_of_longest = pd.NaT

    rows.append({
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'p95_full': p95_val,
        'count_points_in_window': count_points_in_window,
        'count_points_ge_p95_in_window': count_points_ge_p95_in_window,
        'earliest_timestamp_ge_p95_in_window': earliest_ts,
        'max_value_in_window': max_value_in_window,
        'longest_consecutive_run_length_of_values_ge_p95_in_window': longest_run,
        'start_timestamp_of_longest_run_in_window': start_of_longest
    })

# Build result DataFrame
result_df = pd.DataFrame(rows)

# Sort by longest consecutive run desc then by count_points_ge_p95_in_window desc
result_sorted = result_df.sort_values(
    ['longest_consecutive_run_length_of_values_ge_p95_in_window','count_points_ge_p95_in_window'],
    ascending=[False, False]
).reset_index(drop=True)

# Limit to top 50 rows
top50 = result_sorted.head(50)

# Display the compact result
top50

```
Out[5]:
```
Summary (plain English) — metric P95 breaches in the incident window (UTC):

- Overall:
  - I computed global P95 per (service, KPI) on the full metric series, then checked the incident window 2024-01-24 22:44:04 → 2024-01-24 23:14:04 UTC for points >= that P95.
  - Only KPI groups that had at least one point >= their global P95 in the window are shown in the results (37 groups total).

- Important artifacts to note:
  - Several "error" KPIs (e.g., ts-order-service:error and ts-travel-service:error) have global P95 = 0.0. All samples equal 0, so every point meets >= P95 (count_points_ge_p95_in_window = 25). This is an artifact of thresholding on zero-valued series and is not an actionable error (values are zero).
  
- Notable non‑error breaches (examples you should investigate):
  - ts-order-service / diskio
    - p95_full ≈ 58,457; 2 points in the window >= p95
    - earliest breach: 2024-01-24T23:10:00Z
    - max value in window ≈ 68,624
    - longest consecutive run = 2 (start 2024-01-24T23:10:00Z)
  - ts-route-service / latency-50 and latency-90
    - each had 2 points >= p95
    - earliest breaches: 2024-01-24T23:03:00Z
    - max values near the p95 thresholds (small latency spikes)
    - longest consecutive run = 2 (start 2024-01-24T23:03:00Z)
  - ts-travel-service / mem, socket, workload, latency-50/90
    - each had 1–2 breaches in the window; earliest breach timestamps roughly 23:00–23:08Z
    - e.g., mem max in window ≈ 2.9276e8 at 2024-01-24T23:08:00Z; socket max ≈ 21.733 at 23:00:00Z
  - ts-auth-service (from earlier metric checks)
    - CPU and diskio showed brief breaches (2 points each) with earliest CPU breach around 23:01Z and diskio around 22:51Z (recommend verifying exact rows in the table).

- Patterns:
  - Most non-error breaches are short (1–2 consecutive minutes) rather than long sustained outages.
  - Breaches cluster around a few timestamps in the window (notably ~22:47Z, ~22:51Z, ~23:00–23:10Z depending on KPI/service).

- Recommendations (next steps):
  1. Ignore the "error" KPI P95==0 artifact unless error_count > 0 appears; these are zero-valued samples.
  2. Prioritize investigation of the non‑zero breaches with consecutive runs >1:
     - ts-order-service.diskio (23:10Z), ts-route-service.latency (23:03Z), and ts-travel-service memory/socket/workload around 23:00–23:08Z.
  3. Correlate those timestamps with logs and traces (look for increased traffic, GC events, or I/O bursts) and review downstream dependencies.
  4. If you want, I can extract the exact windowed time series rows for any of the flagged (service, KPI) groups to inspect values per minute.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name      p95_full  count_points_in_window  count_points_ge_p95_in_window earliest_timestamp_ge_p95_in_window  max_value_in_window  longest_consecutive_run_length_of_values_ge_p95_in_window start_timestamp_of_longest_run_in_window
0    ts-order-service       error  0.000000e+00                      25                             25           2024-01-24 22:47:00+00:00         0.000000e+00                                                 25                        2024-01-24 22:47:00+00:00
1   ts-travel-service       error  0.000000e+00                      25                             25           2024-01-24 22:47:00+00:00         0.000000e+00                                                 25                        2024-01-24 22:47:00+00:00
2    ts-order-service      diskio  5.845759e+04                      25                              2           2024-01-24 23:10:00+00:00         6.862455e+04                                                  2                        2024-01-24 23:10:00+00:00
3    ts-route-service  latency-50  3.404503e-03                      25                              2           2024-01-24 23:03:00+00:00         3.416018e-03                                                  2                        2024-01-24 23:03:00+00:00
4    ts-route-service  latency-90  7.149150e-03                      25                              2           2024-01-24 23:03:00+00:00         7.207744e-03                                                  2                        2024-01-24 23:03:00+00:00
..                ...         ...           ...                     ...                            ...                                 ...                  ...                                                ...                                              ...
32  ts-travel-service  latency-50  5.667795e-02                      25                              2           2024-01-24 23:02:00+00:00         5.714195e-02                                                  1                        2024-01-24 23:02:00+00:00
33  ts-travel-service  latency-90  4.816498e-01                      25                              2           2024-01-24 23:02:00+00:00         5.466716e-01                                                  1                        2024-01-24 23:02:00+00:00
34  ts-travel-service         mem  2.927369e+08                      25                              2           2024-01-24 23:08:00+00:00         2.927605e+08                                                  1                        2024-01-24 23:08:00+00:00
35  ts-travel-service      socket  1.997000e+01                      25                              2           2024-01-24 23:00:00+00:00         2.173333e+01                                                  1                        2024-01-24 23:00:00+00:00
36  ts-travel-service    workload  1.167936e+01                      25                              2           2024-01-24 23:00:00+00:00         1.201665e+01                                                  1                        2024-01-24 23:00:00+00:00

[37 rows x 9 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# Reuse existing df_trace if available; otherwise load and parse
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')
# ensure timestamp parsed as UTC and value numeric
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# Target substrings to match in trace_name
targets = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']
pattern = '|'.join([t.replace('-', r'\-') for t in targets])  # escape hyphens for regex safety

# Filter trace rows where trace_name mentions any target service (keep any cmdb_id)
df_trace_focus_all = df_trace[df_trace['trace_name'].str.contains(pattern, regex=True, na=False)].copy()

# 1) Full-series aggregates per (cmdb_id, trace_name)
agg_full = (
    df_trace_focus_all
    .groupby(['cmdb_id','trace_name'], dropna=False)['value']
    .agg(
        count_full='count',
        mean_full='mean',
        max_full='max',
        p90_full=lambda s: s.quantile(0.90),
        p95_full=lambda s: s.quantile(0.95),
    )
    .reset_index()
)

# 2) Filter to incident window
window_start = pd.to_datetime("2024-01-24 22:44:04", utc=True)
window_end   = pd.to_datetime("2024-01-24 23:14:04", utc=True)
df_window = df_trace_focus_all[(df_trace_focus_all['timestamp'] >= window_start) & (df_trace_focus_all['timestamp'] <= window_end)].copy()

# Merge p95_full into window rows (thresholds computed on full series)
df_window = df_window.merge(agg_full[['cmdb_id','trace_name','p95_full']], on=['cmdb_id','trace_name'], how='left')

# Mark breaches and error_rate>0
df_window['is_ge_p95'] = df_window['value'] >= df_window['p95_full']
df_window['is_error_rate_gt0'] = df_window['trace_name'].str.contains('error_rate', case=False, na=False) & (df_window['value'] > 0)

# Window-level aggregations
win_agg = (
    df_window
    .groupby(['cmdb_id','trace_name'], dropna=False)
    .agg(
        count_window=('value','count'),
        mean_window=('value','mean'),
        max_window=('value','max'),
        count_points_ge_p95_in_window=('is_ge_p95','sum'),
        error_rate_gt0_count_in_window=('is_error_rate_gt0','sum'),
    )
    .reset_index()
)

# earliest_timestamp_ge_p95_in_window
earliest_ge = df_window[df_window['is_ge_p95']].groupby(['cmdb_id','trace_name'], dropna=False)['timestamp'].min().reset_index(name='earliest_timestamp_ge_p95_in_window')

# timestamp_of_max_window: find min timestamp among rows that equal the group's max_window
max_vals = df_window.groupby(['cmdb_id','trace_name'], dropna=False)['value'].max().reset_index(name='max_window_value')
# merge to find timestamps where value == max_window_value, then take earliest timestamp
max_rows = df_window.merge(max_vals, on=['cmdb_id','trace_name'], how='left')
max_timestamp = max_rows[max_rows['value'] == max_rows['max_window_value']].groupby(['cmdb_id','trace_name'], dropna=False)['timestamp'].min().reset_index(name='timestamp_of_max_window')

# earliest error_rate>0 timestamp
earliest_errgt0 = df_window[df_window['is_error_rate_gt0']].groupby(['cmdb_id','trace_name'], dropna=False)['timestamp'].min().reset_index(name='earliest_error_rate_gt0_timestamp_in_window')

# Merge window aggregates with these timestamps
win_agg = win_agg.merge(earliest_ge, on=['cmdb_id','trace_name'], how='left')
win_agg = win_agg.merge(max_timestamp, on=['cmdb_id','trace_name'], how='left')
win_agg = win_agg.merge(earliest_errgt0, on=['cmdb_id','trace_name'], how='left')

# 3) Combine with full-series p95 and count_full
result = agg_full[['cmdb_id','trace_name','count_full','p95_full']].merge(win_agg, on=['cmdb_id','trace_name'], how='left')

# Fill NaNs for groups missing in the window
result['count_window'] = result['count_window'].fillna(0).astype(int)
result['count_points_ge_p95_in_window'] = result['count_points_ge_p95_in_window'].fillna(0).astype(int)
result['error_rate_gt0_count_in_window'] = result['error_rate_gt0_count_in_window'].fillna(0).astype(int)
result['earliest_timestamp_ge_p95_in_window'] = result['earliest_timestamp_ge_p95_in_window'].where(result['count_points_ge_p95_in_window']>0, pd.NaT)
result['timestamp_of_max_window'] = result['timestamp_of_max_window'].where(result['count_window']>0, pd.NaT)
result['earliest_error_rate_gt0_timestamp_in_window'] = result['earliest_error_rate_gt0_timestamp_in_window'].where(result['error_rate_gt0_count_in_window']>0, pd.NaT)
result['max_window'] = result['max_window'].where(result['count_window']>0, np.nan)

# Select requested columns and sort
result = result[
    ['cmdb_id','trace_name','count_full','p95_full','count_window','count_points_ge_p95_in_window',
     'earliest_timestamp_ge_p95_in_window','max_window','timestamp_of_max_window',
     'error_rate_gt0_count_in_window','earliest_error_rate_gt0_timestamp_in_window']
]

result_sorted = result.sort_values(['count_points_ge_p95_in_window','max_window'], ascending=[False, False]).reset_index(drop=True)

# Limit to top 20 rows
top20_trace_focus = result_sorted.head(20)

# Display compact result
top20_trace_focus

```
Out[6]:
```
Summary of focused trace analysis (plain English):

- Scope: traces whose trace_name mentions any of the target services (ts-auth, ts-order, ts-route, ts-train, ts-travel) for the full series and the incident window 2024-01-24 22:44:04 → 2024-01-24 23:14:04 UTC.

- Key finding: most of the matching trace series in the table are error_rate series with global P95 = 0.0. In the incident window the trace values are also 0.0, so value >= P95 evaluates true and yields count_points_ge_p95_in_window = count_window. However:
  - max_window for these series is 0.0,
  - error_rate_gt0_count_in_window = 0 for all shown rows.
  - In short: there were no non‑zero error_rate events in the window — the P95==0 artifact makes zeros appear as "breaches" under the >= rule but they are not actionable errors.

- No substantive non-zero trace error signals were observed for the target services in the incident window. The result does not show any error_rate > 0 occurrences.

- Recommendation:
  1. Ignore the P95==0 equality artifact for error_rate (treat >0 as the meaningful condition).
  2. To find real issues, inspect non-error trace types (duration, p95 duration, row_count) for the target services and correlate those timestamps with the metric breaches previously found (e.g., auth CPU/diskio spikes, route socket spikes, travel mem/socket spikes around ~22:47, ~22:51, and ~23:00–23:10 UTC).
  3. If you want, I can pull the duration/row_count trace rows for specific service/trace_name around those timestamps for deeper inspection.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                               trace_name  count_full  p95_full  count_window  count_points_ge_p95_in_window earliest_timestamp_ge_p95_in_window  max_window   timestamp_of_max_window  error_rate_gt0_count_in_window earliest_error_rate_gt0_timestamp_in_window
0                      root      trace.to_ts-auth-service.error_rate          25       0.0            25                             25           2024-01-24 22:47:00+00:00         0.0 2024-01-24 22:47:00+00:00                               0                                         NaT
1                      root     trace.to_ts-order-service.error_rate          25       0.0            25                             25           2024-01-24 22:47:00+00:00         0.0 2024-01-24 22:47:00+00:00                               0                                         NaT
2                      root    trace.to_ts-travel-service.error_rate          25       0.0            25                             25           2024-01-24 22:47:00+00:00         0.0 2024-01-24 22:47:00+00:00                               0                                         NaT
3   ts-admin-travel-service    trace.to_ts-travel-service.error_rate          25       0.0            25                             25           2024-01-24 22:47:00+00:00         0.0 2024-01-24 22:47:00+00:00                               0                                         NaT
4           ts-auth-service    trace.from_ts-auth-service.error_rate          25       0.0            25                             25           2024-01-24 22:47:00+00:00         0.0 2024-01-24 22:47:00+00:00                               0                                         NaT
5           ts-auth-service      trace.to_ts-auth-service.error_rate          25       0.0            25                             25           2024-01-24 22:47:00+00:00         0.0 2024-01-24 22:47:00+00:00                               0                                         NaT
6          ts-basic-service     trace.to_ts-route-service.error_rate          25       0.0            25                             25           2024-01-24 22:47:00+00:00         0.0 2024-01-24 22:47:00+00:00                               0                                         NaT
7          ts-basic-service     trace.to_ts-train-service.error_rate          25       0.0            25                             25           2024-01-24 22:47:00+00:00         0.0 2024-01-24 22:47:00+00:00                               0                                         NaT
8           ts-food-service    trace.to_ts-travel-service.error_rate          25       0.0            25                             25           2024-01-24 22:47:00+00:00         0.0 2024-01-24 22:47:00+00:00                               0                                         NaT
9          ts-order-service   trace.from_ts-order-service.error_rate          25       0.0            25                             25           2024-01-24 22:47:00+00:00         0.0 2024-01-24 22:47:00+00:00                               0                                         NaT
10         ts-order-service  trace.from_ts-travel-service.error_rate          25       0.0            25                             25           2024-01-24 22:47:00+00:00         0.0 2024-01-24 22:47:00+00:00                               0                                         NaT
11         ts-order-service     trace.to_ts-order-service.error_rate          25       0.0            25                             25           2024-01-24 22:47:00+00:00         0.0 2024-01-24 22:47:00+00:00                               0                                         NaT
12         ts-route-service   trace.from_ts-route-service.error_rate          25       0.0            25                             25           2024-01-24 22:47:00+00:00         0.0 2024-01-24 22:47:00+00:00                               0                                         NaT
13         ts-route-service  trace.from_ts-travel-service.error_rate          25       0.0            25                             25           2024-01-24 22:47:00+00:00         0.0 2024-01-24 22:47:00+00:00                               0                                         NaT
14         ts-route-service     trace.to_ts-route-service.error_rate          25       0.0            25                             25           2024-01-24 22:47:00+00:00         0.0 2024-01-24 22:47:00+00:00                               0                                         NaT
15          ts-seat-service  trace.from_ts-travel-service.error_rate          25       0.0            25                             25           2024-01-24 22:47:00+00:00         0.0 2024-01-24 22:47:00+00:00                               0                                         NaT
16          ts-seat-service     trace.to_ts-order-service.error_rate          25       0.0            25                             25           2024-01-24 22:47:00+00:00         0.0 2024-01-24 22:47:00+00:00                               0                                         NaT
17          ts-seat-service    trace.to_ts-travel-service.error_rate          25       0.0            25                             25           2024-01-24 22:47:00+00:00         0.0 2024-01-24 22:47:00+00:00                               0                                         NaT
18      ts-security-service     trace.to_ts-order-service.error_rate          25       0.0            25                             25           2024-01-24 22:47:00+00:00         0.0 2024-01-24 22:47:00+00:00                               0                                         NaT
19    ts-ticketinfo-service  trace.from_ts-travel-service.error_rate          25       0.0            25                             25           2024-01-24 22:47:00+00:00         0.0 2024-01-24 22:47:00+00:00                               0                                         NaT```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np

# Reuse existing df_trace if present; otherwise load and parse timestamps/value
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# Candidate service substrings and feature substrings
services = ['ts-auth-service','ts-order-service','ts-route-service','ts-travel-service']
features = ['row_count','duration_mean','duration_p95']

# Build regex patterns
svc_pattern = '|'.join([s.replace('-', r'\-') for s in services])
feat_pattern = '|'.join(features)

# Select trace_names that mention any candidate service AND any desired feature type (based on substring match)
mask_name = df_trace['trace_name'].str.contains(svc_pattern, regex=True, na=False)
mask_feat = df_trace['trace_name'].str.contains(feat_pattern, regex=True, na=False)
df_focus_all = df_trace[mask_name & mask_feat].copy()

# Compute full-series P95 per (cmdb_id, trace_name) using the entire trace.csv (rule: thresholds before window filter)
# Note: compute p95 only for groups present in df_focus_all to limit work
groups_for_threshold = df_focus_all[['cmdb_id','trace_name']].drop_duplicates()
# Merge to ensure thresholds computed from full df_trace for those (cmdb_id,trace_name)
full_thresh = (
    df_trace
    .merge(groups_for_threshold, on=['cmdb_id','trace_name'], how='inner')  # limit to relevant groups but use full-series rows
    .groupby(['cmdb_id','trace_name'], dropna=False)['value']
    .agg(p95_full=lambda s: s.quantile(0.95))
    .reset_index()
)

# Incident window
window_start = pd.to_datetime("2024-01-24 22:44:04", utc=True)
window_end   = pd.to_datetime("2024-01-24 23:14:04", utc=True)

# Filter focused rows to the incident window
df_window = df_focus_all[(df_focus_all['timestamp'] >= window_start) & (df_focus_all['timestamp'] <= window_end)].copy()

# Merge full-series p95 into window rows
df_window = df_window.merge(full_thresh, on=['cmdb_id','trace_name'], how='left')

# Compute indicators
df_window['ge_p95'] = False
mask_has_p95 = ~df_window['p95_full'].isna()
df_window.loc[mask_has_p95, 'ge_p95'] = df_window.loc[mask_has_p95, 'value'] >= df_window.loc[mask_has_p95, 'p95_full']

# Aggregations per (cmdb_id, trace_name) within the window
agg_window = (
    df_window
    .groupby(['cmdb_id','trace_name'], dropna=False)
    .agg(
        count_window=('value','count'),
        mean_window=('value','mean'),
        max_window=('value','max'),
        count_points_ge_p95_in_window=('ge_p95','sum')
    )
    .reset_index()
)

# earliest_timestamp_ge_p95_in_window
earliest_ge = df_window[df_window['ge_p95']].groupby(['cmdb_id','trace_name'])['timestamp'].min().reset_index(name='earliest_timestamp_ge_p95_in_window')

# timestamp_of_max_window: earliest timestamp among rows where value == max_window
max_vals = df_window.groupby(['cmdb_id','trace_name'])['value'].max().reset_index(name='max_val_group')
max_ts = df_window.merge(max_vals, on=['cmdb_id','trace_name'], how='left')
timestamp_of_max = max_ts[max_ts['value'] == max_ts['max_val_group']].groupby(['cmdb_id','trace_name'])['timestamp'].min().reset_index(name='timestamp_of_max_window')

# Combine aggregates and timestamps; include full_series_p95 from full_thresh
result = agg_window.merge(full_thresh, on=['cmdb_id','trace_name'], how='left')
result = result.merge(earliest_ge, on=['cmdb_id','trace_name'], how='left')
result = result.merge(timestamp_of_max, on=['cmdb_id','trace_name'], how='left')

# Clean up columns, ensure types, and fill for groups without breaches
result['earliest_timestamp_ge_p95_in_window'] = result['earliest_timestamp_ge_p95_in_window'].where(result['count_points_ge_p95_in_window']>0, pd.NaT)
result['timestamp_of_max_window'] = result['timestamp_of_max_window'].where(result['count_window']>0, pd.NaT)
result['max_window'] = result['max_window'].where(result['count_window']>0, np.nan)

# Select requested columns and sort
result = result[
    ['cmdb_id','trace_name','count_window','mean_window','max_window','timestamp_of_max_window',
     'p95_full','count_points_ge_p95_in_window','earliest_timestamp_ge_p95_in_window']
]

result_sorted = result.sort_values(['count_points_ge_p95_in_window','max_window'], ascending=[False, False]).reset_index(drop=True)

# Limit to top 20 rows
top20 = result_sorted.head(20)

# Display compact result
top20

```
Out[7]:
```
Summary — focused trace row_count/duration spikes within 2024-01-24 22:44:04 → 2024-01-24 23:14:04 UTC:

What I did:
- Filtered trace series that mention ts-auth-service, ts-order-service, ts-route-service, or ts-travel-service and only kept row_count / duration_mean / duration_p95 features.
- Computed full-series P95 for each (cmdb_id, trace_name) and measured how many window points reached or exceeded that P95, plus peak timestamps in the window.

Top observations (compact):

- Largest count_ge_p95:
  - ts-preserve-service → trace.to_ts-order-service.row_count
    - window count 24, full P95 = 5.0, count >= P95 in window = 5
    - max in window = 6.0 at 2024-01-24T22:47:00Z; earliest >=P95 = 22:47:00Z
- High row_count spikes involving route/travel/order:
  - ts-travel2-service → trace.to_ts-route-service.row_count
    - window count 25, P95 = 564.0, count >= P95 = 3, max = 599 at 22:48:00Z
  - ts-station-service → trace.from_ts-order-service.row_count
    - window count 24, P95 = 10.0, count >= P95 = 3, max = 10 at 22:49:00Z
  - ts-route-service → trace.from/to_ts-route-service.row_count
    - window count 25, P95 = 2059.0, count >= P95 = 2, max = 2156 at 22:48:00Z
  - ts-travel-service → trace.from/to_ts-travel-service.row_count
    - window count 25, P95 ≈ 1774.8, count >= P95 = 2, max = 1819 at 22:47:00Z
  - ts-order-service → trace.from/to_ts-order-service.row_count
    - window count 25, P95 ≈ 516.4, count >= P95 = 2, max = 543 at 22:47:00Z
- Many of these breaches cluster at or just after ~22:47–22:49 UTC (some later peaks like a root trace max at ~23:06 UTC).

Interpretation & next steps:
- The dominant signal here is elevated row_count on specific inter-service traces (order ↔ route ↔ travel interactions) around ~22:47–22:49 UTC and a few later peaks. These indicate short-lived throughput/traffic spikes for those RPCs.
- Correlate these timestamps with:
  - metric breaches already identified (e.g., auth CPU/diskio, route socket, travel mem/socket).
  - application logs for the services involved at the listed earliest timestamps (22:47–22:49 and 23:06).
- If you want deeper inspection, I can extract the raw trace rows (per-minute values) for any of the flagged (cmdb_id, trace_name) groups for the window to see exact per-minute sequences.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                              trace_name  count_window  mean_window  max_window   timestamp_of_max_window  p95_full  count_points_ge_p95_in_window earliest_timestamp_ge_p95_in_window
0     ts-preserve-service     trace.to_ts-order-service.row_count            24     3.083333         6.0 2024-01-24 22:47:00+00:00       5.0                              5           2024-01-24 22:47:00+00:00
1      ts-travel2-service     trace.to_ts-route-service.row_count            25   418.000000       599.0 2024-01-24 22:48:00+00:00     564.0                              3           2024-01-24 22:48:00+00:00
2      ts-station-service   trace.from_ts-order-service.row_count            24     5.916667        10.0 2024-01-24 22:49:00+00:00      10.0                              3           2024-01-24 22:49:00+00:00
3        ts-route-service   trace.from_ts-route-service.row_count            25  1467.080000      2156.0 2024-01-24 22:48:00+00:00    2059.0                              2           2024-01-24 22:48:00+00:00
4        ts-route-service     trace.to_ts-route-service.row_count            25  1467.080000      2156.0 2024-01-24 22:48:00+00:00    2059.0                              2           2024-01-24 22:48:00+00:00
5       ts-travel-service  trace.from_ts-travel-service.row_count            25  1139.920000      1819.0 2024-01-24 22:47:00+00:00    1774.8                              2           2024-01-24 22:47:00+00:00
6       ts-travel-service    trace.to_ts-travel-service.row_count            25  1139.920000      1819.0 2024-01-24 22:47:00+00:00    1774.8                              2           2024-01-24 22:47:00+00:00
7        ts-order-service   trace.from_ts-order-service.row_count            25   334.320000       543.0 2024-01-24 22:47:00+00:00     516.4                              2           2024-01-24 22:47:00+00:00
8        ts-order-service     trace.to_ts-order-service.row_count            25   334.320000       543.0 2024-01-24 22:47:00+00:00     516.4                              2           2024-01-24 22:47:00+00:00
9                    root     trace.to_ts-route-service.row_count            21   391.000000       478.0 2024-01-24 23:06:00+00:00     463.0                              2           2024-01-24 23:06:00+00:00
10  ts-ticketinfo-service  trace.from_ts-travel-service.row_count            25   235.080000       389.0 2024-01-24 22:47:00+00:00     374.6                              2           2024-01-24 22:47:00+00:00
11                   root    trace.to_ts-travel-service.row_count            25   206.600000       375.0 2024-01-24 23:03:00+00:00     350.8                              2           2024-01-24 23:03:00+00:00
12       ts-route-service  trace.from_ts-travel-service.row_count            25   219.640000       336.0 2024-01-24 22:47:00+00:00     333.6                              2           2024-01-24 22:47:00+00:00
13      ts-travel-service     trace.to_ts-route-service.row_count            25   219.640000       336.0 2024-01-24 22:47:00+00:00     333.6                              2           2024-01-24 22:47:00+00:00
14        ts-auth-service    trace.from_ts-auth-service.row_count            25   237.000000       324.0 2024-01-24 22:50:00+00:00     317.2                              2           2024-01-24 22:48:00+00:00
15        ts-auth-service      trace.to_ts-auth-service.row_count            25   237.000000       324.0 2024-01-24 22:50:00+00:00     317.2                              2           2024-01-24 22:48:00+00:00
16        ts-seat-service    trace.to_ts-travel-service.row_count            25   161.240000       275.0 2024-01-24 22:47:00+00:00     261.2                              2           2024-01-24 22:47:00+00:00
17       ts-train-service  trace.from_ts-travel-service.row_count            25   151.960000       244.0 2024-01-24 22:47:00+00:00     241.0                              2           2024-01-24 22:47:00+00:00
18       ts-basic-service     trace.to_ts-route-service.row_count            25    99.880000       147.0 2024-01-24 22:48:00+00:00     144.8                              2           2024-01-24 22:48:00+00:00
19        ts-seat-service     trace.to_ts-order-service.row_count            25    82.000000       138.0 2024-01-24 22:47:00+00:00     130.8                              2           2024-01-24 22:47:00+00:00```
```

In [ ]:
# In[8]:

import pandas as pd

# Reuse df_trace if available; otherwise load and parse timestamps as UTC
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')
    df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Ensure timestamp parsed and value numeric
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# Key minute timestamps (UTC)
key_times = [
    pd.to_datetime("2024-01-24 22:47:00", utc=True),
    pd.to_datetime("2024-01-24 22:48:00", utc=True),
    pd.to_datetime("2024-01-24 23:03:00", utc=True),
    pd.to_datetime("2024-01-24 23:10:00", utc=True),
]

# Candidate cmdb_ids and trace_name substrings
cmdb_candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-travel-service"]
trace_substrings = [
    'to_ts-route-service','to_ts-travel-service','to_ts-order-service',
    'from_ts-travel-service','from_ts-order-service','from_ts-route-service',
    'to_ts-auth-service','from_ts-auth-service'
]
pattern = '|'.join(trace_substrings)

# Filter rows at the exact key timestamps
mask_time = df_trace['timestamp'].isin(key_times)

# Filter rows where cmdb_id is in candidates OR trace_name contains any target substring
mask_cmdb = df_trace['cmdb_id'].isin(cmdb_candidates)
mask_trace_name = df_trace['trace_name'].str.contains(pattern, regex=True, na=False)
mask_service = mask_cmdb | mask_trace_name

df_key = df_trace[mask_time & mask_service].copy()

# Sort by timestamp asc, value desc and select requested columns
df_key_sorted = df_key.sort_values(['timestamp','value'], ascending=[True, False])[
    ['timestamp','cmdb_id','trace_name','value']
].head(20)

df_key_sorted

```
Out[8]:
```
Summary (plain English):

- At the key minute 2024-01-24 22:47:00 UTC the largest trace row_count spikes were:
  - ts-route-service — trace.from_ts-route-service.row_count and trace.to_ts-route-service.row_count = 1972.0 (highest).
  - ts-travel-service — trace.from/to_ts-travel-service.row_count = 1819.0 (second highest).
  - ts-order-service — trace.from/to_ts-order-service.row_count = 543.0 (third).
  - ts-auth-service — trace.from/to_ts-auth-service.row_count = 299.0.
- Many services show high row_count at the same minute calling into the route/travel/order paths (e.g., ts-travel2-service -> to_ts-route_service = 510, ts-ticketinfo_service -> from_ts_travel = 389).
- Interpretation: the earliest and strongest anomaly at 22:47:00 UTC is concentrated on ts-route-service (1972 rows) with ts-route-service also appearing as a downstream target of multiple callers at that time. ts-travel-service and ts-order-service also show large spikes at the same minute.
- Recommendation: investigate ts-route-service and the callers (ts-travel*, ts-order*, ts-basic/ts-travel2) around 22:47:00 UTC (check traces, logs, and metrics) as the likely downstream hotspot at the start of the incident.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp                cmdb_id                                trace_name   value
479 2024-01-24 22:47:00+00:00       ts-route-service     trace.from_ts-route-service.row_count  1972.0
491 2024-01-24 22:47:00+00:00       ts-route-service       trace.to_ts-route-service.row_count  1972.0
667 2024-01-24 22:47:00+00:00      ts-travel-service    trace.from_ts-travel-service.row_count  1819.0
691 2024-01-24 22:47:00+00:00      ts-travel-service      trace.to_ts-travel-service.row_count  1819.0
327 2024-01-24 22:47:00+00:00       ts-order-service     trace.from_ts-order-service.row_count   543.0
347 2024-01-24 22:47:00+00:00       ts-order-service       trace.to_ts-order-service.row_count   543.0
487 2024-01-24 22:47:00+00:00       ts-route-service   trace.from_ts-travel2-service.row_count   510.0
719 2024-01-24 22:47:00+00:00     ts-travel2-service       trace.to_ts-route-service.row_count   510.0
611 2024-01-24 22:47:00+00:00  ts-ticketinfo-service    trace.from_ts-travel-service.row_count   389.0
683 2024-01-24 22:47:00+00:00      ts-travel-service  trace.to_ts-ticketinfo-service.row_count   389.0
483 2024-01-24 22:47:00+00:00       ts-route-service    trace.from_ts-travel-service.row_count   336.0
675 2024-01-24 22:47:00+00:00      ts-travel-service       trace.to_ts-route-service.row_count   336.0
127 2024-01-24 22:47:00+00:00        ts-auth-service      trace.from_ts-auth-service.row_count   299.0
131 2024-01-24 22:47:00+00:00        ts-auth-service        trace.to_ts-auth-service.row_count   299.0
535 2024-01-24 22:47:00+00:00        ts-seat-service      trace.to_ts-travel-service.row_count   275.0
663 2024-01-24 22:47:00+00:00      ts-travel-service      trace.from_ts-seat-service.row_count   275.0
635 2024-01-24 22:47:00+00:00       ts-train-service    trace.from_ts-travel-service.row_count   244.0
687 2024-01-24 22:47:00+00:00      ts-travel-service       trace.to_ts-train-service.row_count   244.0
151 2024-01-24 22:47:00+00:00       ts-basic-service       trace.to_ts-route-service.row_count   140.0
475 2024-01-24 22:47:00+00:00       ts-route-service     trace.from_ts-basic-service.row_count   140.0```
```